In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import os
import glob
import gzip
import warnings
from io import BytesIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import folium
import gpxpy
from fitparse import FitFile
from haversine import haversine, Unit
from pyproj import Transformer
from datetime import timezone, timedelta
from IPython.display import display, clear_output
from IPython.display import HTML

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
# ── Configuratie ────────────────────────────────────────────────────────────
PERSONEN = {
    'Jan': [
        '/workspaces/case4_strava/data/StravaJan',
        '/workspaces/case4_strava/data/StravaJan/activities_dump_download',
    ],
    'Pieter': [
        '/workspaces/case4_strava/data/StravaPieter',
        '/workspaces/case4_strava/data/StravaPieter/Alle activiteiten',
    ],
    'Robin': [
        '/workspaces/case4_strava/data/StravaRobin',
        '/workspaces/case4_strava/data/StravaRobin/Alle activiteiten',
    ],
}

LOCAL_TZ = timezone(timedelta(hours=2))  # zomertijd (CEST)
# LOCAL_TZ = timezone(timedelta(hours=1))  # wintertijd (CET)

MAX_SNELHEID_KMH = 70.0

In [ ]:
# ── Inladen per persoon met uitgebreide logging ─────────────────────────────
fit_per_persoon = verzamel_fit_per_persoon(PERSONEN)

activiteiten_per_persoon = {}
inventaris = []  # voor overzichtstabel achteraf

for persoon, paden in fit_per_persoon.items():
    geladen = []
    for pad in paden:
        bestand = os.path.basename(pad)
        try:
            df = laad_fit(pad)
            if not df.empty:
                df['persoon'] = persoon
                geladen.append(df)
                inventaris.append({
                    'persoon': persoon,
                    'bestand': bestand,
                    'status':  'geladen',
                    'reden':   '',
                })
            else:
                inventaris.append({
                    'persoon': persoon,
                    'bestand': bestand,
                    'status':  'overgeslagen',
                    'reden':   'geen GPS',
                })
        except Exception as e:
            inventaris.append({
                'persoon': persoon,
                'bestand': bestand,
                'status':  'fout',
                'reden':   str(e),
            })

    activiteiten_per_persoon[persoon] = geladen
    n_geladen    = sum(1 for r in inventaris if r['persoon'] == persoon and r['status'] == 'geladen')
    n_geen_gps   = sum(1 for r in inventaris if r['persoon'] == persoon and r['reden']  == 'geen GPS')
    n_fout       = sum(1 for r in inventaris if r['persoon'] == persoon and r['status'] == 'fout')
    print(f'{persoon}: {n_geladen} geladen | {n_geen_gps} geen GPS | {n_fout} fouten')

totaal = sum(len(v) for v in activiteiten_per_persoon.values())
print(f'\nTotaal geladen: {totaal} .fit activiteiten')

# ── Overzichtstabel ─────────────────────────────────────────────────────────
df_inventaris = pd.DataFrame(inventaris)

print('\nSamenvatting per persoon en status:')
display(df_inventaris.groupby(['persoon', 'status']).size().unstack(fill_value=0))

# Toon alle niet-geladen bestanden voor inspectie
niet_geladen = df_inventaris[df_inventaris['status'] != 'geladen']
if not niet_geladen.empty:
    print(f'\nNiet-geladen bestanden ({len(niet_geladen)} totaal):')
    display(niet_geladen[['persoon', 'bestand', 'status', 'reden']].reset_index(drop=True))

In [ ]:
import json
def kolom_inventaris(activiteiten: list, persoon: str) -> pd.DataFrame:
    kolom_stats = {}
    n_activiteiten = len(activiteiten)

    for df in activiteiten:
        # Verwijder duplicate kolomnamen voor de berekening
        df_uniek = df.loc[:, ~df.columns.duplicated()]
        
        for kolom in df_uniek.columns:
            if kolom not in kolom_stats:
                kolom_stats[kolom] = {'aanwezig': 0, 'niet_null': []}
            kolom_stats[kolom]['aanwezig'] += 1
            pct_niet_null = float(df_uniek[kolom].notna().mean() * 100)
            kolom_stats[kolom]['niet_null'].append(pct_niet_null)

    rijen = []
    for kolom, stats in kolom_stats.items():
        rijen.append({
            'persoon':           persoon,
            'kolom':             kolom,
            'aanwezig_in':       stats['aanwezig'],
            'aanwezig_pct':      round(stats['aanwezig'] / n_activiteiten * 100, 1),
            'gem_niet_null_pct': round(float(np.mean(stats['niet_null'])), 1),
        })

    return pd.DataFrame(rijen).sort_values('aanwezig_pct', ascending=False).reset_index(drop=True)

def exporteer_activiteiten_json(activiteiten: list, persoon: str, uitvoerpad: str):
    export = {
        'persoon':        persoon,
        'n_activiteiten': len(activiteiten),
        'activiteiten':   []
    }

    for df in activiteiten:
        # Verwijder duplicate kolomnamen
        df = df.loc[:, ~df.columns.duplicated()]
        
        kolommen = {}
        for kolom in df.columns:
            niet_null = df[kolom].dropna()
            kolommen[kolom] = {
                'dtype':         str(df[kolom].dtype),
                'n_waarden':     int(df[kolom].notna().sum()),
                'niet_null_pct': round(float(df[kolom].notna().mean() * 100), 1),
                'sample':        [str(v) for v in niet_null.head(3).tolist()] if not niet_null.empty else []
            }

        export['activiteiten'].append({
            'bestand':  df['bron'].iloc[0] if 'bron' in df.columns else 'onbekend',
            'sport':    df['sport'].iloc[0] if 'sport' in df.columns else 'onbekend',
            'n_punten': len(df),
            'start':    str(df['timestamp'].min()) if 'timestamp' in df.columns else None,
            'eind':     str(df['timestamp'].max()) if 'timestamp' in df.columns else None,
            'kolommen': kolommen,
        })

    with open(uitvoerpad, 'w', encoding='utf-8') as f:
        json.dump(export, f, indent=2, ensure_ascii=False)

    print(f'✅ JSON opgeslagen: {uitvoerpad}')
# ── Uitvoeren ────────────────────────────────────────────────────────────────
inventaris_per_persoon = {}

for persoon, activiteiten in activiteiten_per_persoon.items():
    # Kolom-inventarisatie
    inventaris_per_persoon[persoon] = kolom_inventaris(activiteiten, persoon)
    print(f'\n=== {persoon} ({len(activiteiten)} activiteiten) ===')
    display(inventaris_per_persoon[persoon])

    # JSON export
    exporteer_activiteiten_json(
        activiteiten,
        persoon,
        uitvoerpad=f'kolommen_{persoon.lower()}.json'
    )

# ── Vergelijking over alle personen ─────────────────────────────────────────
print('\n=== Aanwezigheid per kolom over alle personen ===')
df_vergelijk = pd.concat(inventaris_per_persoon.values())
df_pivot = df_vergelijk.pivot_table(
    index='kolom',
    columns='persoon',
    values='aanwezig_pct',
    aggfunc='first'
).fillna(0).round(1)

df_pivot['gemiddelde'] = df_pivot.mean(axis=1).round(1)
df_pivot = df_pivot.sort_values('gemiddelde', ascending=False)

display(df_pivot.style
    .background_gradient(cmap='RdYlGn', subset=[p for p in PERSONEN.keys()])
    .format('{:.1f}%')
)

In [ ]:
# ── Activiteitstype per persoon visualiseren ────────────────────────────────

# Bouw een samenvattingstabel op uit de geladen activiteiten
activiteit_records = []
for persoon, activiteiten in activiteiten_per_persoon.items():
    for df in activiteiten:
        activiteit_records.append({
            'persoon': persoon,
            'sport':   df['sport'].iloc[0] if 'sport' in df.columns else 'onbekend',
        })

df_activiteiten = pd.DataFrame(activiteit_records)

# Tel activiteiten per persoon per sport
df_counts = (df_activiteiten
    .groupby(['persoon', 'sport'])
    .size()
    .reset_index(name='aantal')
    .sort_values(['persoon', 'aantal'], ascending=[True, False])
)

# ── Plot ─────────────────────────────────────────────────────────────────────
personen = list(activiteiten_per_persoon.keys())
n_personen = len(personen)

fig, axes = plt.subplots(1, n_personen, figsize=(6 * n_personen, 6))
if n_personen == 1:
    axes = [axes]

kleuren = plt.cm.Set2.colors

for ax, persoon in zip(axes, personen):
    data = df_counts[df_counts['persoon'] == persoon].reset_index(drop=True)
    
    bars = ax.barh(
        data['sport'],
        data['aantal'],
        color=[kleuren[i % len(kleuren)] for i in range(len(data))],
        edgecolor='white',
        linewidth=1.2
    )
    
    # Waarde labels op de bars
    for bar, aantal in zip(bars, data['aantal']):
        ax.text(
            bar.get_width() + 0.5,
            bar.get_y() + bar.get_height() / 2,
            str(aantal),
            va='center', ha='left', fontsize=10, fontweight='bold'
        )
    
    ax.set_title(persoon, fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Aantal activiteiten')
    ax.set_ylabel('')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0, data['aantal'].max() * 1.15)

plt.suptitle('Activiteiten per persoon', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('activiteiten_per_persoon.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Grafiek opgeslagen als activiteiten_per_persoon.png')